In [2]:
import pypsa
import pandas as pd
import numpy as np

Parse the datasets from entsoe for total load, pv generation and energy prices (selected date is 15.07.2025)

In [3]:
prices_csv = pd.read_csv("entsoe_datasets/energy_prices_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
prices_csv = prices_csv[prices_csv["Sequence"] == "Sequence 1"]
print(prices_csv)

pv_generation_csv = pd.read_csv("entsoe_datasets/pv_generation_15_07_2025.csv",
                            parse_dates=["time"],
                            index_col=["time"],
                            usecols=lambda col: "time" in col or "Day-ahead" in col,
                            date_format="%Y-%m-%d %H:%M:%S")
print(pv_generation_csv)

load_csv = pd.read_csv("entsoe_datasets/load_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
print(load_csv)

                       Sequence  Day-ahead Price (EUR/MWh)
time                                                      
15/07/2025 00:00:00  Sequence 1                     106.10
15/07/2025 01:00:00  Sequence 1                     103.06
15/07/2025 02:00:00  Sequence 1                      93.86
15/07/2025 03:00:00  Sequence 1                      88.28
15/07/2025 04:00:00  Sequence 1                      87.12
15/07/2025 05:00:00  Sequence 1                      93.00
15/07/2025 06:00:00  Sequence 1                     110.33
15/07/2025 07:00:00  Sequence 1                     113.41
15/07/2025 08:00:00  Sequence 1                     106.86
15/07/2025 09:00:00  Sequence 1                      90.82
15/07/2025 10:00:00  Sequence 1                      77.67
15/07/2025 11:00:00  Sequence 1                      65.00
15/07/2025 12:00:00  Sequence 1                      59.19
15/07/2025 13:00:00  Sequence 1                      47.26
15/07/2025 14:00:00  Sequence 1                      40.

Create a PyPSA network and set the snapshots for a period of 24 h.

In [4]:
network = pypsa.Network()
start_time = "2025-07-15 00:00:00"
timestamps = pd.date_range(start_time, periods=24, freq="h")
network.set_snapshots(timestamps)

Add the buses for the IEEE 33-bus system. The base voltage is 12.66 kV

In [5]:
for i in range(1, 34):
    network.add(
        "Bus",
        f"Bus_{i}",
        v_nom=12.66
    )

Adding the lines. Resistance (r) and reactance (x) are in Ohm. The thermal capacity (s_nom) is set high so as to not be a limiting factor

In [6]:
# (from, to, r, x)
line_data = [
    (1, 2, 0.0922, 0.047),      (2, 3, 0.493, 0.2511),      (3, 4, 0.366, 0.1864), 
    (4, 5, 0.3811, 0.1941),     (5, 6, 0.819, 0.707),       (6, 7, 0.1872, 0.6188), 
    (7, 8, 0.7114, 0.2351),     (8, 9, 1.03, 0.74),         (9, 10, 1.044, 0.74),   
    (10, 11, 0.1966, 0.065),    (11, 12, 0.3744, 0.198),    (12, 13, 1.468, 1.155),
    (13, 14, 0.5416, 0.7129),   (14, 15, 0.591, 0.526),     (15, 16, 0.7463, 0.545),
    (16, 17, 1.289, 1.721),     (17, 18, 0.732, 0.574),     (2, 19, 0.164, 0.1565),
    (19, 20, 1.5042, 1.3554),   (20, 21, 0.4095, 0.4784),   (21, 22, 0.7089, 0.9373),   
    (3, 23, 0.4512, 0.3083),    (23, 24, 0.898, 0.7091),    (24, 25, 0.896, 0.7011), 
    (6, 26, 0.203, 0.1034),     (26, 27, 0.2842, 0.1447),   (27, 28, 1.059, 0.9337), 
    (28, 29, 0.8042, 0.7006),   (29, 30, 0.5075, 0.2585),   (30, 31, 0.9744, 0.963), 
    (31, 32, 0.3105, 0.3619),   (32, 33, 0.341, 0.5302),
]

for i, (from_b, to_b, r, x) in enumerate(line_data):
    network.add(
        "Line", 
        f"Line_{from_b}-{to_b}",
        bus0=f"Bus_{from_b}",
        bus1=f"Bus_{to_b}",
        r=r,
        x=x,
        s_nom=5000
    )

Add the loads. p_set is the active power (kW) and q_set is the reactive power (kVAr)

In [7]:
load_data = [
    (100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]

Define all the profiles an normalize them

In [9]:
raw_load_series = load_csv.values
normalized_load_series = raw_load_series / raw_load_series.max()
load_series = []
for load in normalized_load_series:
    load_series.append(load)
# print(load_series)
# print("a\n")

raw_pv_series = pv_generation_csv.values
normalized_pv_series = raw_pv_series / raw_pv_series.max()
pv_series = []
for pv in normalized_pv_series:
    pv_series.append(pv)
print(pv_series)

price_series = []
for price in prices_csv.values:
    price_series.append(price[1])
# price_series = 
print(price_series)

[array([0.]), array([0.]), array([0.]), array([0.]), array([0.]), array([0.00094907]), array([0.04285548]), array([0.17001653]), array([0.37162545]), array([0.58918418]), array([0.76882737]), array([0.89537788]), array([0.96876131]), array([1.]), array([0.98545738]), array([0.93643815]), array([0.83896498]), array([0.69460114]), array([0.49686438]), array([0.27330249]), array([0.10469043]), array([0.01367431]), array([0.]), array([0.])]
[106.1, 103.06, 93.86, 88.28, 87.12, 93.0, 110.33, 113.41, 106.86, 90.82, 77.67, 65.0, 59.19, 47.26, 40.2, 42.57, 65.0, 84.87, 101.54, 108.15, 119.78, 141.99, 121.28, 112.45]


Apply the time varying profiles

In [ ]:
for i, (p, q) in enumerate(load_data):
    real_load_profile = np.array(load_series) * p

    # load_profile_series = pd.Series(real_load_profile, index=network.snapshots)
    network.add(
        "Load",
        f"Load_bus_{i+2}",
        bus=f"Bus_{i+2}",
        p_set=real_load_profile,
        q_set=q
    )

Add grid, PV and batteries

In [ ]:
network.add(
    "Generator",
    "Substation",
    bus="Bus_1",
    p_nom=10000,
    p_min_pu=-1.0,
    carrier="gas",
    marginal_cost=price_series
)

for i in range(2, 34):

    # raw_pv_col = pv_cols[i % len(pv_cols)]
    # raw_pv = dataset_day[raw_pv_col].fillna(0)
    # raw_pv_series = raw_pv.values
    # real_pv_profile = raw_pv_series / raw_pv_series.max()
    # # real_pv_profile = real_pv_profile.fillna(0)

    network.add(
        "Generator",
        f"PV_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=150,
        p_max_pu=pv_series,
        carrier="solar",
        marginal_cost=0
    )
    network.add(
        "StorageUnit",
        f"Battery_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=200, # Nominal capacity in kW
        max_hours=4, # Energy capacity is p_nom * max_hours = 800 kWh
        carrier="battery",
        efficiency_store=0.9,
        efficiency_dispatch=0.9,
        standing_loss=0.01 # 1% loss per hour
    )

In [ ]:
network.optimize()

In [ ]:
print(f"cost: {network.objective:.2f}")


In [ ]:
network.generators_t.p

In [ ]:
network.storage_units_t.p

In [ ]:
network.loads_t.p
print("a")
load_tot = []
for i in range(24):
    load_0 = network.loads_t.p.Load_bus_2.values[i] 
    + network.loads_t.p.Load_bus_3.values[i] 
    + network.loads_t.p.Load_bus_4.values[i] 
    + network.loads_t.p.Load_bus_5.values[i]
    + network.loads_t.p.Load_bus_6.values[i] 
    + network.loads_t.p.Load_bus_7.values[i] 
    + network.loads_t.p.Load_bus_8.values[i]
    + network.loads_t.p.Load_bus_9.values[i] 
    + network.loads_t.p.Load_bus_10.values[i] 
    + network.loads_t.p.Load_bus_11.values[i]
    + network.loads_t.p.Load_bus_12.values[i] 
    + network.loads_t.p.Load_bus_13.values[i] 
    + network.loads_t.p.Load_bus_14.values[i]
    + network.loads_t.p.Load_bus_15.values[i] 
    + network.loads_t.p.Load_bus_16.values[i] 
    + network.loads_t.p.Load_bus_17.values[i]
    + network.loads_t.p.Load_bus_18.values[i] 
    + network.loads_t.p.Load_bus_19.values[i] 
    + network.loads_t.p.Load_bus_20.values[i]
    + network.loads_t.p.Load_bus_21.values[i] 
    + network.loads_t.p.Load_bus_22.values[i] 
    + network.loads_t.p.Load_bus_23.values[i]
    + network.loads_t.p.Load_bus_24.values[i] 
    + network.loads_t.p.Load_bus_25.values[i] 
    + network.loads_t.p.Load_bus_26.values[i]
    + network.loads_t.p.Load_bus_27.values[i] 
    + network.loads_t.p.Load_bus_28.values[i] 
    + network.loads_t.p.Load_bus_29.values[i]
    + network.loads_t.p.Load_bus_30.values[i] 
    + network.loads_t.p.Load_bus_31.values[i] 
    + network.loads_t.p.Load_bus_32.values[i]
    + network.loads_t.p.Load_bus_33.values[i] 
    print(load_0)